# Unsupervised Anomaly Detection Practice

**Topic:** Clustering and PCA Review &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_test` `(3000, 12)`: the readings to audit, ~3% anomalous.
- `X_clean` `(4000, 12)`: a reference batch of verified normal readings you may use to learn what "normal" looks like.
- The true anomaly flags are created with the data and kept out of your way. The evaluation cell scores your predictions with F1.

## Your task

1. Learn the structure of normal data from `X_clean` (cluster it, fit PCA, or fit `IsolationForest` / `EllipticEnvelope`).
2. Score every row of `X_test` for abnormality and flag the most abnormal ~3%.
3. Set `predictions` to a `(3000,)` array of 0 (normal) / 1 (anomaly).

## Evaluation

F1 score on the anomaly class must be ≥ 0.6.


In [ ]:
import numpy as np

rng = np.random.default_rng(311)
d, k_lat = 12, 4
mix = rng.normal(0, 1.0, (k_lat, d))     # normal data lives near a 4-D subspace

def normal_batch(n, seed):
    r = np.random.default_rng(seed)
    return r.normal(size=(n, k_lat)) @ mix + r.normal(0, 0.35, (n, d))

X_clean = normal_batch(4000, 321)

_n_test, _n_anom = 3000, 90
_normal = normal_batch(_n_test - _n_anom, 322)
_anom = rng.normal(0, 2.6, (_n_anom, d))          # off-structure readings
X_test = np.vstack([_normal, _anom])
_y_anom = np.concatenate([np.zeros(_n_test - _n_anom, int), np.ones(_n_anom, int)])
_shuf = rng.permutation(_n_test)
X_test, _y_anom = X_test[_shuf], _y_anom[_shuf]

print(f"X_clean {X_clean.shape}, X_test {X_test.shape} (~3% anomalies)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: F1 on the hidden anomaly flags
# =====================================================================
import numpy as np

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel().astype(int)
if _pred.shape != _y_anom.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_y_anom.shape}.")

_tp = int(((_pred == 1) & (_y_anom == 1)).sum())
_fp = int(((_pred == 1) & (_y_anom == 0)).sum())
_fn = int(((_pred == 0) & (_y_anom == 1)).sum())
_prec = _tp / max(1, _tp + _fp)
_rec = _tp / max(1, _tp + _fn)
_f1 = 0.0 if _prec + _rec == 0 else 2 * _prec * _rec / (_prec + _rec)
print("=" * 52)
print(f"Flagged {int(_pred.sum())} of {len(_pred)} rows as anomalies (true count: {int(_y_anom.sum())})")
print(f"Precision {_prec:.3f} | Recall {_rec:.3f} | F1 {_f1:.3f}   (target: >= 0.6)")
print("=" * 52)
print("PASSED: anomalies found!" if _f1 >= 0.6 else "NOT PASSED yet: try PCA reconstruction error on X_clean.")


## Solution

Normal readings live near a 4-D subspace, so fit PCA on `X_clean` and score each test row by its reconstruction error: anomalies off the subspace reconstruct badly. Flag the top ~3%.

```python
import numpy as np

mu = X_clean.mean(axis=0)
_, _, Vt = np.linalg.svd(X_clean - mu, full_matrices=False)
components = Vt[:4]

def recon_error(X):
    Z = (X - mu) @ components.T
    back = Z @ components + mu
    return np.linalg.norm(X - back, axis=1)

scores = recon_error(X_test)
threshold = np.quantile(scores, 0.97)
predictions = (scores >= threshold).astype(int)
print(f"flagged {int(predictions.sum())} rows")
```
